# Stage-2 6-Class Hard-Negative Crop Verifier (`ConvNeXt-Tiny` v2)

Trains a **6-class** Stage-2 crop verifier (`other_bg`, `phone`, `power_tool`, `table_bg`, `umbrella`, `weapon`) on Google Colab T4 GPU (~12-18 min) to eliminate false alarms on cordless drills and horizontally aimed dark umbrellas — **with zero data leakage** against the 37-clip evaluation benchmark.

**Pipeline:**
1. Restore cached base crops from Google Drive (or auto-download from public OpenImages V7 if no cache exists).
2. Download public **Google OpenImages V7** hard negatives (`Drill (Tool)`, `Flashlight`, `Hair dryer`, extra `Umbrella`, `Mobile phone`, `Remote control`) using `0.30` crop padding matching `src/perception.py`.
3. Train `ConvNeXt-Tiny` with differential learning rates, mixed precision (`FP16`), 90/270-degree orientation augmentation for distractors, CCTV JPEG/blur degradation, image-grouped train/val splits, and per-epoch atomic Drive checkpoints.
4. Verify strict `torch.load(..., weights_only=True)` compatibility, print 6-class metrics & confusion matrix, and download `crop_verifier_convnext_tiny_v2.pt` to place in `weights/`.

In [ ]:
# Cell 1: GPU Check, Google Drive Mount & Crash-Safe Save Helper
import os
from pathlib import Path
import torch

!nvidia-smi
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Switch Runtime -> Change runtime type -> T4 GPU.")

from google.colab import drive
drive.mount("/content/drive")

OLD_DRIVE_CACHE = Path("/content/drive/MyDrive/weapon_v2_runs/verifier_crops_cache.zip")
DRIVE_RUNS = Path("/content/drive/MyDrive/weapon_detector_runs")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

V2_CROPS_ZIP = DRIVE_RUNS / "verifier_v2_crops_cache.zip"
WORK_DIR = Path("/content/work")
CROPS_DIR = WORK_DIR / "verifier_v2_crops"
META_DIR = WORK_DIR / "oid_meta"
CROPS_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)


def atomic_torch_save(obj, dst_path: Path) -> None:
    dst = Path(dst_path)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, dst)


def show_crop_counts() -> dict[str, int]:
    counts = {}
    for d in sorted(CROPS_DIR.iterdir()):
        if d.is_dir():
            counts[d.name] = len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png")))
    print("Current crop counts:", counts)
    return counts

In [ ]:
# Cell 2: Restore Existing Cache from Drive (Fast Path) or Flag Full OpenImages Bootstrap
import zipfile

FORCE_REBUILD_CROPS = False  # Set True only to wipe and re-download everything

restored_from = None
if not FORCE_REBUILD_CROPS:
    if V2_CROPS_ZIP.is_file():
        restored_from = V2_CROPS_ZIP
    elif OLD_DRIVE_CACHE.is_file():
        restored_from = OLD_DRIVE_CACHE

if restored_from is not None and not any(CROPS_DIR.glob("*/*.jpg")):
    print(f"[Cache Hit] Extracting {restored_from} -> {CROPS_DIR}...")
    with zipfile.ZipFile(restored_from, "r") as zf:
        zf.extractall(CROPS_DIR)
elif any(CROPS_DIR.glob("*/*.jpg")):
    print("[Skip] Crops already present in", CROPS_DIR)
else:
    print("[Info] No Drive crop cache found — Cell 3 will bootstrap all 6 classes from OpenImages V7.")

counts = show_crop_counts()
NEED_BASE_FROM_OPENIMAGES = counts.get("weapon", 0) < 100
print("NEED_BASE_FROM_OPENIMAGES =", NEED_BASE_FROM_OPENIMAGES)

In [ ]:
# Cell 3: Fetch Zero-Leakage Public Google OpenImages V7 Hard Negatives
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
import os
from pathlib import Path
import random
import threading
import urllib.request
import zipfile

import cv2
import numpy as np
import requests

SEED = 42
random.seed(SEED)

# Quotas are measured for newly fetched OpenImages `oid_*` crops
TARGET_SPEC = [
    ("power_tool", ["Drill (Tool)", "Drill"], 600),
    ("power_tool", ["Flashlight"], 250),
    ("power_tool", ["Hair dryer"], 150),
    ("umbrella", ["Umbrella"], 400),
    ("phone", ["Mobile phone"], 200),
    ("phone", ["Remote control"], 200),
]

if NEED_BASE_FROM_OPENIMAGES:
    print("[Bootstrap Mode] Adding weapon, table_bg, and other_bg OpenImages V7 quotas...")
    TARGET_SPEC.extend([
        ("weapon", ["Handgun"], 700),
        ("weapon", ["Rifle"], 500),
        ("weapon", ["Shotgun"], 300),
        ("table_bg", ["Table"], 500),
        ("other_bg", ["Backpack"], 250),
        ("other_bg", ["Handbag"], 250),
    ])


def download_if_missing(url: str, dst: Path) -> Path:
    if dst.is_file() and dst.stat().st_size > 0:
        return dst
    print(f"[Download] {url} -> {dst.name}")
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    urllib.request.urlretrieve(url, tmp)
    os.replace(tmp, dst)
    return dst


CLASS_CSV = download_if_missing(
    "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",
    META_DIR / "oidv7-class-descriptions-boxable.csv",
)

display_to_mid: dict[str, str] = {}
with CLASS_CSV.open("r", encoding="utf-8", newline="") as fh:
    for row in csv.reader(fh):
        if len(row) < 2:
            continue
        mid, disp = row[0].strip(), row[1].strip()
        if mid.lower() == "labelname":
            continue
        display_to_mid[disp.lower()] = mid

# Build target MID -> (bucket, target_display, remaining_quota)
mid_plan: dict[str, dict] = {}
for bucket, aliases, target_n in TARGET_SPEC:
    matched_mid = None
    matched_name = None
    for alias in aliases:
        if alias.lower() in display_to_mid:
            matched_mid = display_to_mid[alias.lower()]
            matched_name = alias
            break
    if matched_mid is None:
        print(f"[Warn] Could not find OpenImages MID for aliases={aliases}")
        continue

    existing_oid = len(list((CROPS_DIR / bucket).glob("oid_*.jpg"))) if (CROPS_DIR / bucket).is_dir() else 0
    bucket_total_needed = sum(q for b, _, q in TARGET_SPEC if b == bucket)
    if existing_oid >= int(0.85 * bucket_total_needed):
        print(f"[Cache] Bucket '{bucket}' already has {existing_oid} OpenImages crops (target={bucket_total_needed}).")
        continue

    mid_plan[matched_mid] = {
        "bucket": bucket,
        "display": matched_name,
        "quota": target_n,
        "boxes": [],  # list of (split, image_id, xmin, xmax, ymin, ymax)
    }
    print(f"  Target MID {matched_mid} ({matched_name} -> {bucket}) quota={target_n}")

seen_img_bucket: dict[tuple[str, str, str], int] = {}  # (split, image_id, bucket) -> box_count


def all_quotas_filled() -> bool:
    return all(len(info["boxes"]) >= info["quota"] for info in mid_plan.values())


def scan_bbox_rows(reader: csv.DictReader, split_name: str, max_pool_mult: int = 2) -> None:
    if not mid_plan or all_quotas_filled():
        return
    for row in reader:
        mid = row.get("LabelName", "")
        if mid not in mid_plan:
            continue
        info = mid_plan[mid]
        if len(info["boxes"]) >= info["quota"] * max_pool_mult:
            if all(len(v["boxes"]) >= v["quota"] for v in mid_plan.values()):
                break
            continue
        if row.get("IsDepiction", "0") == "1" or row.get("IsGroupOf", "0") == "1":
            continue
        try:
            xmin = float(row["XMin"])
            xmax = float(row["XMax"])
            ymin = float(row["YMin"])
            ymax = float(row["YMax"])
        except (KeyError, TypeError, ValueError):
            continue
        if (xmax - xmin) < 0.03 or (ymax - ymin) < 0.03 or xmax <= xmin or ymax <= ymin:
            continue

        img_id = row["ImageID"].strip()
        key = (split_name, img_id, info["bucket"])
        if seen_img_bucket.get(key, 0) >= 2:
            continue
        seen_img_bucket[key] = seen_img_bucket.get(key, 0) + 1
        info["boxes"].append((split_name, img_id, xmin, xmax, ymin, ymax))

        if all(len(v["boxes"]) >= v["quota"] * max_pool_mult for v in mid_plan.values()):
            break


if mid_plan:
    val_csv = download_if_missing(
        "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",
        META_DIR / "validation-annotations-bbox.csv",
    )
    with val_csv.open("r", encoding="utf-8", newline="") as fh:
        scan_bbox_rows(csv.DictReader(fh), "validation", max_pool_mult=1)

    if not all_quotas_filled():
        test_csv = download_if_missing(
            "https://storage.googleapis.com/openimages/v5/test-annotations-bbox.csv",
            META_DIR / "test-annotations-bbox.csv",
        )
        with test_csv.open("r", encoding="utf-8", newline="") as fh:
            scan_bbox_rows(csv.DictReader(fh), "test", max_pool_mult=1)

    if not all_quotas_filled():
        needed = {m: f"{i['display']}({len(i['boxes'])}/{i['quota']})" for m, i in mid_plan.items() if len(i["boxes"]) < i["quota"]}
        print("[Stream] Streaming oidv6-train-annotations-bbox.csv (stops immediately once quotas fill) for:", needed)
        train_url = "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv"
        with requests.get(train_url, stream=True, timeout=120) as resp:
            resp.raise_for_status()
            lines_iter = (line for line in resp.iter_lines(decode_unicode=True) if line)
            header = next(csv.reader([next(lines_iter)]))
            col_idx = {h: i for i, h in enumerate(header)}
            req_cols = ("ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax")
            assert all(c in col_idx for c in req_cols), f"Unexpected header: {header}"
            dep_i = col_idx.get("IsDepiction", -1)
            grp_i = col_idx.get("IsGroupOf", -1)

            for r_i, row in enumerate(csv.reader(lines_iter), start=1):
                if r_i % 2_000_000 == 0:
                    prog = {i["display"]: f"{len(i['boxes'])}/{i['quota']}" for i in mid_plan.values() if len(i["boxes"]) < i["quota"]}
                    print(f"  ...scanned {r_i // 1_000_000}M train bbox rows | remaining={prog}")
                if len(row) < len(header):
                    continue
                mid = row[col_idx["LabelName"]]
                if mid not in mid_plan:
                    continue
                info = mid_plan[mid]
                if len(info["boxes"]) >= info["quota"]:
                    if all_quotas_filled():
                        break
                    continue
                if (dep_i >= 0 and row[dep_i] == "1") or (grp_i >= 0 and row[grp_i] == "1"):
                    continue
                try:
                    xmin = float(row[col_idx["XMin"]])
                    xmax = float(row[col_idx["XMax"]])
                    ymin = float(row[col_idx["YMin"]])
                    ymax = float(row[col_idx["YMax"]])
                except ValueError:
                    continue
                if (xmax - xmin) < 0.03 or (ymax - ymin) < 0.03 or xmax <= xmin or ymax <= ymin:
                    continue
                img_id = row[col_idx["ImageID"]].strip()
                key = ("train", img_id, info["bucket"])
                if seen_img_bucket.get(key, 0) >= 2:
                    continue
                seen_img_bucket[key] = seen_img_bucket.get(key, 0) + 1
                info["boxes"].append(("train", img_id, xmin, xmax, ymin, ymax))
                if all_quotas_filled():
                    break

    # Group downloads by (split, img_id) so an image with 2 boxes is downloaded once
    jobs_by_image: dict[tuple[str, str], list[tuple[str, int, float, float, float, float]]] = {}
    for mid, info in mid_plan.items():
        boxes = info["boxes"]
        random.shuffle(boxes)
        selected = boxes[: info["quota"]]
        print(f"  Queued {len(selected)}/{info['quota']} boxes for {info['display']} -> {info['bucket']}")
        for idx, (split_name, img_id, xmin, xmax, ymin, ymax) in enumerate(selected):
            jobs_by_image.setdefault((split_name, img_id), []).append(
                (info["bucket"], idx, xmin, xmax, ymin, ymax)
            )

    sess_lock = threading.Lock()
    saved_counts: dict[str, int] = {}
    dl_failed = 0

    def process_image_job(split_name: str, img_id: str, box_specs: list) -> int:
        url = f"https://open-images-dataset.s3.amazonaws.com/{split_name}/{img_id}.jpg"
        try:
            r = requests.get(url, timeout=20)
            if r.status_code != 200 or not r.content:
                return 0
            arr = np.frombuffer(r.content, dtype=np.uint8)
            img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            if img is None:
                return 0
        except Exception:
            return 0

        fh, fw = img.shape[:2]
        n_ok = 0
        for bucket, idx, xmin, xmax, ymin, ymax in box_specs:
            x1 = max(0, min(fw - 1, int(round(xmin * fw))))
            y1 = max(0, min(fh - 1, int(round(ymin * fh))))
            x2 = max(x1 + 1, min(fw, int(round(xmax * fw))))
            y2 = max(y1 + 1, min(fh, int(round(ymax * fh))))
            bw, bh = x2 - x1, y2 - y1
            if bw < 24 or bh < 24:
                continue

            # Exact 0.30 pad_ratio matching src/perception.py::_extract_padded_crop
            px1 = max(0, x1 - int(round(bw * 0.30)))
            py1 = max(0, y1 - int(round(bh * 0.30)))
            px2 = min(fw, x2 + int(round(bw * 0.30)))
            py2 = min(fh, y2 + int(round(bh * 0.30)))
            crop = img[py1:py2, px1:px2]
            if crop.size == 0:
                continue

            interp = cv2.INTER_AREA if (crop.shape[0] > 256 or crop.shape[1] > 256) else cv2.INTER_LINEAR
            resized = cv2.resize(crop, (256, 256), interpolation=interp)

            out_dir = CROPS_DIR / bucket
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / f"oid_{split_name}_{img_id}_b{idx}.jpg"
            cv2.imwrite(str(out_file), resized, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
            with sess_lock:
                saved_counts[bucket] = saved_counts.get(bucket, 0) + 1
            n_ok += 1
        return n_ok

    print(f"[Fetch] Downloading {len(jobs_by_image)} unique OpenImages photos (16 threads)...")
    with ThreadPoolExecutor(max_workers=16) as pool:
        futs = [pool.submit(process_image_job, sp, iid, bxs) for (sp, iid), bxs in jobs_by_image.items()]
        done = 0
        for fut in as_completed(futs):
            done += 1
            if fut.result() == 0:
                dl_failed += 1
            if done % 300 == 0 or done == len(futs):
                print(f"  Progress: {done}/{len(futs)} photos | saved={saved_counts} | skipped/empty={dl_failed}")

# Save unified 6-class cache atomically to Drive so future runs restore in seconds
counts = show_crop_counts()
if any(counts.values()):
    tmp_zip = V2_CROPS_ZIP.with_suffix(".zip.tmp")
    print(f"[Cache] Saving 6-class crops archive -> {V2_CROPS_ZIP}...")
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_STORED) as zf:
        for cls_dir in sorted(CROPS_DIR.iterdir()):
            if not cls_dir.is_dir():
                continue
            for img_p in sorted(cls_dir.glob("*.jpg")):
                zf.write(img_p, arcname=f"{cls_dir.name}/{img_p.name}")
    os.replace(tmp_zip, V2_CROPS_ZIP)
    print(f"[Saved] {V2_CROPS_ZIP} ({V2_CROPS_ZIP.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Cell 4: Train 6-Class ConvNeXt-Tiny v2 Verifier (Image-Grouped Split, Pure OpenCV+Torchvision Augs, Per-Epoch Resume)
from collections import defaultdict
import copy
import os
from pathlib import Path
import random
import re

import cv2
import numpy as np
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 15
BACKBONE_LR = 1.5e-4
HEAD_LR = 6.0e-4
WEIGHT_DECAY = 1e-2
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

OUT_CKPT = DRIVE_RUNS / "crop_verifier_convnext_tiny_v2.pt"
RESUME_CKPT = DRIVE_RUNS / "verifier_v2_resume.pt"
FORCE_RESTART_VERIFIER = False  # Set True only to restart training from Epoch 1


def extract_group_key(path: Path) -> str:
    stem = path.stem
    m_coco = re.match(r"^(coco_[a-z_]+_\d{6,})_\d+$", stem)
    if m_coco:
        return m_coco.group(1)
    return re.sub(r"_(b\d+|crop\d+|box\d+|det\d+)$", "", stem)


def build_grouped_split(root: Path, val_ratio: float = 0.15) -> tuple[list[str], list[tuple[str, int]], list[tuple[str, int]]]:
    classes = sorted([d.name for d in root.iterdir() if d.is_dir() and len(list(d.glob("*.jpg"))) >= 10])
    assert "weapon" in classes, f"Expected 'weapon' folder in {root}, got {classes}"
    assert "power_tool" in classes, f"Expected 'power_tool' folder in {root}, got {classes}"
    cls_to_idx = {c: i for i, c in enumerate(classes)}

    train_rel, val_rel = [], []
    rng = random.Random(SEED)
    for c in classes:
        files = sorted((root / c).glob("*.jpg"))
        groups: dict[str, list[Path]] = defaultdict(list)
        for p in files:
            groups[extract_group_key(p)].append(p)
        keys = sorted(groups.keys())
        rng.shuffle(keys)
        n_val_groups = max(1, int(round(len(keys) * val_ratio)))
        val_keys = set(keys[:n_val_groups])
        for k in keys:
            target_list = val_rel if k in val_keys else train_rel
            for p in groups[k]:
                target_list.append((p.relative_to(root).as_posix(), cls_to_idx[c]))
    return classes, train_rel, val_rel


def augment_bgr(img: np.ndarray, cls_name: str, is_train: bool) -> np.ndarray:
    if not is_train:
        return cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

    h, w = img.shape[:2]
    # 1. Horizontal flip
    if random.random() < 0.5:
        img = cv2.flip(img, 1)

    # 2. Strong 90/270 orientation augmentation on handheld/elongated distractors
    if cls_name in ("power_tool", "umbrella", "phone") and random.random() < 0.35:
        rot_code = cv2.ROTATE_90_CLOCKWISE if random.random() < 0.5 else cv2.ROTATE_90_COUNTERCLOCKWISE
        img = cv2.rotate(img, rot_code)
        h, w = img.shape[:2]

    # 3. Mild affine rotation & scale
    if random.random() < 0.65:
        angle = random.uniform(-25.0, 25.0)
        scale = random.uniform(0.88, 1.12)
        mat = cv2.getRotationMatrix2D((w * 0.5, h * 0.5), angle, scale)
        img = cv2.warpAffine(img, mat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

    # 4. Brightness / contrast / gamma jitter (dark shadows / sunlit outdoor variance)
    if random.random() < 0.55:
        alpha = random.uniform(0.75, 1.28)
        beta = random.uniform(-24.0, 24.0)
        img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        if random.random() < 0.4:
            gamma = random.uniform(0.80, 1.25)
            lut = np.clip(((np.arange(256) / 255.0) ** gamma) * 255.0, 0, 255).astype(np.uint8)
            img = cv2.LUT(img, lut)

    # 5. Mild CCTV blur / motion blur
    if random.random() < 0.30:
        if random.random() < 0.5:
            k = random.choice((3, 5))
            img = cv2.GaussianBlur(img, (k, k), 0)
        else:
            k = random.choice((3, 5))
            kernel = np.zeros((k, k), dtype=np.float32)
            kernel[k // 2, :] = 1.0 / k
            img = cv2.filter2D(img, -1, kernel)

    # 6. Realistic CCTV JPEG compression artifacts
    if random.random() < 0.45:
        q = random.randint(35, 88)
        ok, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), q])
        if ok:
            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
            if dec is not None:
                img = dec

    return cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)


class VerifierCropDataset(Dataset):
    def __init__(self, items: list[tuple[Path, int]], classes: list[str], is_train: bool):
        self.items = items
        self.classes = classes
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        path, label = self.items[idx]
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = augment_bgr(img, self.classes[label], is_train=self.is_train)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        rgb = (rgb - MEAN) / STD
        tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).contiguous()
        return tensor, label


resume_data = None
if RESUME_CKPT.is_file() and not FORCE_RESTART_VERIFIER:
    resume_data = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
    classes = list(resume_data["classes"])
    train_rel = list(resume_data["train_rel"])
    val_rel = list(resume_data["val_rel"])
    train_items = [(CROPS_DIR / rel, y) for rel, y in train_rel if (CROPS_DIR / rel).is_file()]
    val_items = [(CROPS_DIR / rel, y) for rel, y in val_rel if (CROPS_DIR / rel).is_file()]
    if len(train_items) == 0 or len(val_items) == 0:
        print("[Warn] Resume paths missing from /content/work — building fresh grouped split.")
        resume_data = None

if resume_data is None:
    classes, train_rel, val_rel = build_grouped_split(CROPS_DIR, val_ratio=0.15)
    train_items = [(CROPS_DIR / rel, y) for rel, y in train_rel]
    val_items = [(CROPS_DIR / rel, y) for rel, y in val_rel]

print(f"Classes ({len(classes)}): {classes}")
print(f"Grouped split -> train={len(train_items)} | val={len(val_items)}")

train_counts = np.bincount([y for _, y in train_items], minlength=len(classes)).astype(np.float32)
inv_w = len(train_items) / (len(classes) * np.maximum(train_counts, 1.0))
inv_w = inv_w / inv_w.mean()
inv_w = np.clip(inv_w, 0.5, 2.5)
print("Class weights:", {c: round(float(w), 3) for c, w in zip(classes, inv_w)})

train_ds = VerifierCropDataset(train_items, classes, is_train=True)
val_ds = VerifierCropDataset(val_items, classes, is_train=False)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=use_cuda, persistent_workers=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=use_cuda, persistent_workers=True
)

model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(classes))
model = model.to(device)

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(inv_w, dtype=torch.float32, device=device),
    label_smoothing=0.05,
)

start_epoch = 1
best_score = -1e9
best_auc = 0.0
best_drill_fp = 1.0
best_umb_fp = 1.0
best_state_dict = None

if resume_data is not None:
    model.load_state_dict(resume_data["model_state"])
    optimizer.load_state_dict(resume_data["optimizer_state"])
    scheduler.load_state_dict(resume_data["scheduler_state"])
    scaler.load_state_dict(resume_data["scaler_state"])
    start_epoch = int(resume_data["epoch"]) + 1
    best_score = float(resume_data.get("best_score", -1e9))
    best_auc = float(resume_data.get("best_auc", 0.0))
    best_drill_fp = float(resume_data.get("best_drill_fp", 1.0))
    best_umb_fp = float(resume_data.get("best_umb_fp", 1.0))
    best_state_dict = resume_data.get("best_state_dict")
    print(f"[Resume] Continuing from epoch {start_epoch}/{EPOCHS} (best_auc={best_auc:.4f})")

weapon_idx = classes.index("weapon")
drill_idx = classes.index("power_tool")
umb_idx = classes.index("umbrella")

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_cuda):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tr_loss += float(loss.item()) * len(y)
    tr_loss /= max(1, len(train_ds))
    scheduler.step()

    model.eval()
    v_loss = 0.0
    all_probs, all_targets = [], []
    with torch.inference_mode():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=use_cuda):
                logits = model(x)
                loss = criterion(logits, y)
            v_loss += float(loss.item()) * len(y)
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(y.cpu().numpy())

    v_loss /= max(1, len(val_ds))
    probs_np = np.concatenate(all_probs, axis=0)
    targets_np = np.concatenate(all_targets, axis=0)
    preds_np = probs_np.argmax(axis=1)
    val_acc = float((preds_np == targets_np).mean())

    p_weapon = probs_np[:, weapon_idx]
    auc = float(roc_auc_score((targets_np == weapon_idx).astype(np.int32), p_weapon))

    w_mask = targets_np == weapon_idx
    d_mask = targets_np == drill_idx
    u_mask = targets_np == umb_idx
    w_rec = float((p_weapon[w_mask] >= 0.52).mean()) if w_mask.any() else 0.0
    drill_fp = float((p_weapon[d_mask] >= 0.52).mean()) if d_mask.any() else 0.0
    umb_fp = float((p_weapon[u_mask] >= 0.52).mean()) if u_mask.any() else 0.0

    score = auc - 0.15 * max(0.0, drill_fp - 0.03) - 0.10 * max(0.0, umb_fp - 0.03)

    mark = ""
    if score > best_score or best_state_dict is None:
        best_score = score
        best_auc = auc
        best_drill_fp = drill_fp
        best_umb_fp = umb_fp
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        atomic_torch_save(
            {
                "arch": "convnext_tiny",
                "classes": [str(c) for c in classes],
                "weapon_classes": ["weapon"],
                "img_size": int(IMG_SIZE),
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225],
                "state_dict": best_state_dict,
                "val_weapon_auc": float(best_auc),
                "val_power_tool_fp_rate": float(best_drill_fp),
                "val_umbrella_fp_rate": float(best_umb_fp),
            },
            OUT_CKPT,
        )
        mark = " *BEST*"

    atomic_torch_save(
        {
            "epoch": epoch,
            "classes": classes,
            "train_rel": train_rel,
            "val_rel": val_rel,
            "best_score": best_score,
            "best_auc": best_auc,
            "best_drill_fp": best_drill_fp,
            "best_umb_fp": best_umb_fp,
            "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_state_dict": best_state_dict,
        },
        RESUME_CKPT,
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | tr_loss={tr_loss:.4f} | val_loss={v_loss:.4f} | "
        f"acc={val_acc*100:.1f}% | weapon_auc={auc:.4f} | w_rec@0.52={w_rec*100:.1f}% | "
        f"power_tool_fp@0.52={drill_fp*100:.1f}% | umb_fp@0.52={umb_fp*100:.1f}%{mark}"
    )

print(f"\n[Done] Best checkpoint saved to {OUT_CKPT} (auc={best_auc:.4f}, drill_fp={best_drill_fp*100:.2f}%, umb_fp={best_umb_fp*100:.2f}%)")

In [ ]:
# Cell 5: Verify `weights_only=True` Load, Print 6-Class Report + Confusion Matrix & Download
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny

AUTO_DOWNLOAD = True

ckpt = torch.load(OUT_CKPT, map_location="cpu", weights_only=True)
loaded_classes = [str(c) for c in ckpt["classes"]]
print("Verified `weights_only=True` load! File size:", round(OUT_CKPT.stat().st_size / 1e6, 2), "MB")
print("Metadata:", {k: v for k, v in ckpt.items() if k != "state_dict"})

verify_model = convnext_tiny(weights=None)
verify_model.classifier[2] = nn.Linear(verify_model.classifier[2].in_features, len(loaded_classes))
verify_model.load_state_dict(ckpt["state_dict"], strict=True)
verify_model = verify_model.to(device).eval()
if use_cuda:
    verify_model = verify_model.half()

all_probs, all_targets = [], []
with torch.inference_mode():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        if use_cuda:
            x = x.half()
        probs = torch.softmax(verify_model(x).float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(y.numpy())

probs_np = np.concatenate(all_probs, axis=0)
targets_np = np.concatenate(all_targets, axis=0)
preds_np = probs_np.argmax(axis=1)

w_idx = loaded_classes.index("weapon")
p_weapon = probs_np[:, w_idx]
auc = roc_auc_score((targets_np == w_idx).astype(np.int32), p_weapon)

print(f"\n=== Final Validation Report (FP16 GPU Inference, Weapon-vs-All ROC-AUC = {auc:.4f}) ===")
print(classification_report(targets_np, preds_np, target_names=loaded_classes, digits=4))
print("Confusion Matrix (rows=true, cols=pred):")
print("Classes order:", loaded_classes)
print(confusion_matrix(targets_np, preds_np))

for distractor in ("power_tool", "umbrella", "phone", "table_bg", "other_bg"):
    if distractor in loaded_classes:
        idx = loaded_classes.index(distractor)
        mask = targets_np == idx
        fp52 = float((p_weapon[mask] >= 0.52).mean()) if mask.any() else 0.0
        print(f"  {distractor:12s} FP@0.52 = {fp52*100:.2f}% ({int((p_weapon[mask] >= 0.52).sum())}/{int(mask.sum())})")

print("\nGoogle Drive path:", OUT_CKPT)
print("Save downloaded file locally as: weights/crop_verifier_convnext_tiny_v2.pt")

if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(OUT_CKPT))
    except Exception as exc:
        print("Browser download skipped/timed out (safe — download directly from Google Drive instead):", exc)